# Gemma 3 1B BF16 LoRA fine-tuning profiles

이 노트북은 한 개의 `RUN_PROFILE`로 네 가지 실험을 선택한다.

- `quick_smoke_128`: 40-step 코드 경로 확인
- `memorize_128`: 128개를 충분히 학습할 수 있는지 확인하는 과적합 진단
- `pilot_1k`: 1,000개에서 train/validation 분리 학습
- `generalize_10k`: 10,000개에서 첫 일반화 품질 확인

공통 검증 항목은 BF16 로딩, loss 기록, peak VRAM, adapter 저장·재로드다.

> 현재 corpus는 Minecraft QA이며 Odyssey actor trajectory가 아니다. `memorize_128`의 성공은 학습 구현의 정상 작동만 뜻하고 actor 성능을 입증하지 않는다.

In [1]:
from pathlib import Path
import gc
import json
import math
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import transformers
import accelerate
import peft
import trl
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "MineMA-Model-Fine-Tuning").is_dir() and (candidate / "research").is_dir():
            return candidate
    raise FileNotFoundError("MineSkynet repository root를 찾지 못했습니다.")


ROOT = find_repo_root(Path.cwd())
MODEL_DIR = ROOT / "MineMA-Model-Fine-Tuning/models/gemma-3-1b-it"
DATA_DIR = ROOT / "research/mineskynet_finetuning/data"
SMOKE_TRAIN_FILE = DATA_DIR / "splits/smoke_train_128.jsonl"
FULL_TRAIN_FILE = DATA_DIR / "splits/train.parquet"
VALID_FILE = DATA_DIR / "splits/validation.parquet"
OUTPUTS_ROOT = ROOT / "research/mineskynet_finetuning/outputs"

SEED = 42
set_seed(SEED)

print("Repository:", ROOT)
print("Model:", MODEL_DIR)
print("Data:", DATA_DIR)

/home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: /home/pluto2479/Documents/MineSkynet
Model: /home/pluto2479/Documents/MineSkynet/MineMA-Model-Fine-Tuning/models/gemma-3-1b-it
Data: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data


## 1. 환경·모델·데이터 사전 검사

CUDA/BF16 지원, 필수 로컬 파일, 모델 architecture와 패키지 버전을 검사한다. 여기서 실패하면 학습을 시작하지 않는다.

In [2]:
required_model_files = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "generation_config.json",
]
missing_model_files = [name for name in required_model_files if not (MODEL_DIR / name).is_file()]

assert torch.cuda.is_available(), "CUDA GPU를 찾지 못했습니다."
assert torch.cuda.is_bf16_supported(), "현재 GPU/PyTorch 조합은 BF16을 지원하지 않습니다."
assert MODEL_DIR.is_dir(), f"모델 디렉터리가 없습니다: {MODEL_DIR}"
assert not missing_model_files, f"모델 파일이 없습니다: {missing_model_files}"
assert SMOKE_TRAIN_FILE.is_file(), f"smoke train 파일이 없습니다: {SMOKE_TRAIN_FILE}"
assert FULL_TRAIN_FILE.is_file(), f"full train 파일이 없습니다: {FULL_TRAIN_FILE}"
assert VALID_FILE.is_file(), f"validation 파일이 없습니다: {VALID_FILE}"

model_config = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
model_bytes = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file())

preflight = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "peft": peft.__version__,
    "trl": trl.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_gib": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "model_type": model_config.model_type,
    "architectures": model_config.architectures,
    "model_disk_gib": round(model_bytes / 1024**3, 2),
}
print(json.dumps(preflight, ensure_ascii=False, indent=2))

{
  "timestamp_utc": "2026-08-05T21:53:55.268103+00:00",
  "python": "3.10.20",
  "torch": "2.12.1+cu130",
  "torch_cuda": "13.0",
  "transformers": "5.14.1",
  "accelerate": "1.14.0",
  "peft": "0.20.0",
  "trl": "1.9.2",
  "gpu": "NVIDIA GeForce RTX 3090",
  "gpu_total_gib": 23.56,
  "bf16_supported": true,
  "model_type": "gemma3_text",
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "model_disk_gib": 1.9
}


## 2. Base Gemma BF16 로딩과 학습 전 추론

`local_files_only=True`로 다운로드 무결성을 확인한다. Gemma의 `generation_config.json`에 정의된 `<eos>`와 `<end_of_turn>`을 모두 보존하기 위해 `eos_token_id`를 수동으로 덮어쓰지 않는다.

In [3]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
assert tokenizer.pad_token_id is not None

tokenizer.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=torch.bfloat16,
    local_files_only=True,
).to("cuda")
model.config.use_cache = False

CHECK_MESSAGES = [{
    "role": "user",
    "content": "In Minecraft, how many wooden planks are required to craft a crafting table?",
}]


def generate_answer(current_model, messages, max_new_tokens=64):
    current_model.eval()
    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(current_model.device)
    prompt_length = model_inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output_ids = current_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(
        output_ids[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()


base_answer = generate_answer(model, CHECK_MESSAGES)
base_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
print("Base answer:", base_answer)
print(f"Base load+inference peak VRAM: {base_peak_gib:.2f} GiB")
assert base_answer, "Base 모델이 빈 응답을 생성했습니다."

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 14266.48it/s]


Base answer: You need **6** wooden planks to craft a crafting table in Minecraft. 

(1 wood plank x 6 = 6 planks)
Base load+inference peak VRAM: 1.92 GiB


## 3. 실행 프로필 선택

아래 셀의 `RUN_PROFILE` 한 줄만 바꾼다. 각 프로필은 별도 출력 폴더를 사용하므로 adapter가 서로 덮어써지지 않는다.

`memorize_128`은 train 데이터를 그대로 eval에도 사용한다. 이 수치는 일반화 성능이 아니라 암기 가능성만 측정한다.

In [4]:
RUN_PROFILE = "generalize_10k"

PROFILES = {
    "quick_smoke_128": {
        "purpose": "코드 경로와 adapter 저장을 빠르게 확인",
        "train_source": "smoke",
        "train_rows": 128,
        "eval_rows": 32,
        "eval_on_train": False,
        "max_length": 512,
        "max_steps": 40,
        "num_train_epochs": 1.0,
        "batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "eval_steps": 10,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
    },
    "memorize_128": {
        "purpose": "128개 과적합으로 학습 구현과 LoRA 용량 진단",
        "train_source": "smoke",
        "train_rows": 128,
        "eval_rows": 128,
        "eval_on_train": True,
        "max_length": 512,
        "max_steps": 200,
        "num_train_epochs": 1.0,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "eval_steps": 20,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.0,
    },
    "pilot_1k": {
        "purpose": "1,000개에서 작은 train/validation 일반화 시험",
        "train_source": "full",
        "train_rows": 1_000,
        "eval_rows": 200,
        "eval_on_train": False,
        "max_length": 512,
        "max_steps": -1,
        "num_train_epochs": 3.0,
        "batch_size": 8,
        "gradient_accumulation_steps": 2,
        "learning_rate": 1e-4,
        "eval_steps": 25,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    "generalize_10k": {
        "purpose": "10,000개에서 첫 품질·일반화 평가",
        "train_source": "full",
        "train_rows": 10_000,
        "eval_rows": 500,
        "eval_on_train": False,
        "max_length": 512,
        "max_steps": -1,
        "num_train_epochs": 2.0,
        "batch_size": 8,
        "gradient_accumulation_steps": 2,
        "learning_rate": 1e-4,
        "eval_steps": 100,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
}

assert RUN_PROFILE in PROFILES, f"알 수 없는 profile: {RUN_PROFILE}"
profile = PROFILES[RUN_PROFILE].copy()
MAX_LENGTH = profile["max_length"]
OUTPUT_DIR = OUTPUTS_ROOT / f"gemma3_1b_lora_{RUN_PROFILE}"
ADAPTER_DIR = OUTPUT_DIR / "adapter"

effective_batch = profile["batch_size"] * profile["gradient_accumulation_steps"]
if profile["max_steps"] > 0:
    estimated_steps = profile["max_steps"]
else:
    steps_per_epoch = math.ceil(profile["train_rows"] / effective_batch)
    estimated_steps = math.ceil(steps_per_epoch * profile["num_train_epochs"])

profile_report = {
    "run_profile": RUN_PROFILE,
    **profile,
    "effective_batch": effective_batch,
    "estimated_optimizer_steps": estimated_steps,
    "output_dir": str(OUTPUT_DIR),
}
print(json.dumps(profile_report, ensure_ascii=False, indent=2))

{
  "run_profile": "generalize_10k",
  "purpose": "10,000개에서 첫 품질·일반화 평가",
  "train_source": "full",
  "train_rows": 10000,
  "eval_rows": 500,
  "eval_on_train": false,
  "max_length": 512,
  "max_steps": -1,
  "num_train_epochs": 2.0,
  "batch_size": 8,
  "gradient_accumulation_steps": 2,
  "learning_rate": 0.0001,
  "eval_steps": 100,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "effective_batch": 16,
  "estimated_optimizer_steps": 1250,
  "output_dir": "/home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/outputs/gemma3_1b_lora_generalize_10k"
}


## 4. 프로필에 맞는 데이터 준비

`full` 프로필은 고정 seed로 train·validation에서 각각 표본을 뽑는다. `memorize_128`만 의도적으로 train과 eval이 같고, 나머지는 전처리 단계에서 분리한 validation을 사용한다.

In [5]:
expected_columns = ["instruction", "input", "output"]

if profile["train_source"] == "smoke":
    source_train_df = pd.read_json(SMOKE_TRAIN_FILE, lines=True)
else:
    source_train_df = pd.read_parquet(FULL_TRAIN_FILE, columns=expected_columns)

assert list(source_train_df.columns) == expected_columns
assert len(source_train_df) >= profile["train_rows"]
train_df = source_train_df.sample(
    n=profile["train_rows"],
    random_state=SEED,
).reset_index(drop=True)

if profile["eval_on_train"]:
    valid_df = train_df.copy()
else:
    source_valid_df = pd.read_parquet(VALID_FILE, columns=expected_columns)
    assert len(source_valid_df) >= profile["eval_rows"]
    valid_df = source_valid_df.sample(
        n=profile["eval_rows"],
        random_state=SEED,
    ).reset_index(drop=True)

assert not train_df[["instruction", "output"]].isna().any().any()
assert not valid_df[["instruction", "output"]].isna().any().any()


def user_content(row):
    instruction = str(row["instruction"]).strip()
    extra_input = str(row.get("input", "")).strip()
    if extra_input:
        return f"{instruction}\n\nInput:\n{extra_input}"
    return instruction


def to_prompt_completion(frame: pd.DataFrame) -> Dataset:
    records = []
    for row in frame.to_dict(orient="records"):
        records.append({
            "prompt": [{"role": "user", "content": user_content(row)}],
            "completion": [{"role": "assistant", "content": str(row["output"]).strip()}],
        })
    return Dataset.from_list(records)


train_dataset = to_prompt_completion(train_df)
eval_dataset = to_prompt_completion(valid_df)


def rendered_length(example):
    messages = example["prompt"] + example["completion"]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
    )
    return len(encoded["input_ids"])


lengths = np.array([rendered_length(row) for row in train_dataset])
length_report = {
    "profile": RUN_PROFILE,
    "train_rows": len(train_dataset),
    "eval_rows": len(eval_dataset),
    "eval_on_train": profile["eval_on_train"],
    "min_tokens": int(lengths.min()),
    "median_tokens": int(np.median(lengths)),
    "p95_tokens": int(np.percentile(lengths, 95)),
    "max_tokens": int(lengths.max()),
    "rows_over_max_length": int((lengths > MAX_LENGTH).sum()),
}
print(json.dumps(length_report, ensure_ascii=False, indent=2))
print("Example:", train_dataset[0])

{
  "profile": "generalize_10k",
  "train_rows": 10000,
  "eval_rows": 500,
  "eval_on_train": false,
  "min_tokens": 17,
  "median_tokens": 54,
  "p95_tokens": 135,
  "max_tokens": 1780,
  "rows_over_max_length": 1
}
Example: {'prompt': [{'role': 'user', 'content': 'What distinguishes cave air from regular air in Minecraft?'}], 'completion': [{'role': 'assistant', 'content': 'Cave air in Minecraft is distinguished from regular air by its location and generation pattern. It is found within carver caves, underground structures, and certain biomes like badlands and near lava lakes. While cave air functions identically to regular air in terms of gameplay mechanics, its presence indicates underground spaces and natural formations within the game world.'}]}


## 5. LoRA와 Trainer 생성

첫 실험들은 양자화 없이 BF16으로 학습한다. Gemma chat template의 turn 종료 토큰을 명시하고 completion 영역에만 loss를 적용한다.

In [6]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=profile["lora_r"],
    lora_alpha=profile["lora_alpha"],
    lora_dropout=profile["lora_dropout"],
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

training_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_steps=profile["max_steps"],
    num_train_epochs=profile["num_train_epochs"],
    per_device_train_batch_size=profile["batch_size"],
    gradient_accumulation_steps=profile["gradient_accumulation_steps"],
    per_device_eval_batch_size=profile["batch_size"],
    learning_rate=profile["learning_rate"],
    warmup_steps=max(1, round(estimated_steps * 0.1)),
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LENGTH,
    packing=False,
    completion_only_loss=True,
    eos_token="<end_of_turn>",
    eval_strategy="steps",
    eval_steps=profile["eval_steps"],
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()
print("READY TO TRAIN:", RUN_PROFILE)
print("Estimated optimizer steps:", estimated_steps)
print("Output directory:", OUTPUT_DIR)

Truncating train dataset: 100%|██████████| 10000/10000 [00:00<00:00, 23254.77 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 10000/10000 [00:00<00:00, 35972.72 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 500/500 [00:00<00:00, 34265.51 examples/s]


trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879
READY TO TRAIN: generalize_10k
Estimated optimizer steps: 1250
Output directory: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/outputs/gemma3_1b_lora_generalize_10k


## 6. 학습 실행

이 셀부터 실제 GPU 학습이 시작된다. 별도 터미널에서 `watch -n 1 nvidia-smi`로 관찰할 수 있다. 이전 profile 산출물과는 별도 폴더에 저장된다.

In [7]:
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train()
train_peak_gib = torch.cuda.max_memory_allocated() / 1024**3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

loss_rows = [
    {"step": row["step"], "loss": row["loss"]}
    for row in trainer.state.log_history
    if "loss" in row and math.isfinite(row["loss"])
]
assert loss_rows, "기록된 finite train loss가 없습니다."

window = min(5, len(loss_rows))
first_window = np.mean([row["loss"] for row in loss_rows[:window]])
last_window = np.mean([row["loss"] for row in loss_rows[-window:]])
summary = {
    "run_profile": RUN_PROFILE,
    "profile": profile,
    "first_loss_window_mean": float(first_window),
    "last_loss_window_mean": float(last_window),
    "loss_decreased": bool(last_window < first_window),
    "peak_vram_gib": round(train_peak_gib, 3),
    "adapter_dir": str(ADAPTER_DIR),
    "train_metrics": train_result.metrics,
}
(OUTPUT_DIR / "run_profile.json").write_text(
    json.dumps(profile_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "smoke_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert ADAPTER_DIR.joinpath("adapter_config.json").is_file()
assert summary["loss_decreased"], "마지막 loss 평균이 최초 평균보다 감소하지 않았습니다."

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.060321,2.026083,1.929809,104400.000000,0.573443
200,1.884462,1.929071,1.925386,205342.000000,0.584359
300,1.645396,1.868500,1.794021,308236.000000,0.591305
400,1.862615,1.830100,1.723274,410664.000000,0.598575
500,1.985684,1.814996,1.655310,514228.000000,0.598418
600,1.641067,1.782744,1.657553,619154.000000,0.604617
700,1.507128,1.782418,1.528092,723642.000000,0.607507
800,1.355399,1.766348,1.540443,829509.000000,0.610985
900,1.529154,1.762432,1.513882,931453.000000,0.612043
1000,1.135818,1.753370,1.521623,1031793.000000,0.612582


{
  "run_profile": "generalize_10k",
  "profile": {
    "purpose": "10,000개에서 첫 품질·일반화 평가",
    "train_source": "full",
    "train_rows": 10000,
    "eval_rows": 500,
    "eval_on_train": false,
    "max_length": 512,
    "max_steps": -1,
    "num_train_epochs": 2.0,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 0.0001,
    "eval_steps": 100,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05
  },
  "first_loss_window_mean": 4.967551136016846,
  "last_loss_window_mean": 1.7024847269058228,
  "loss_decreased": true,
  "peak_vram_gib": 4.232,
  "adapter_dir": "/home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/outputs/gemma3_1b_lora_generalize_10k/adapter",
  "train_metrics": {
    "train_runtime": 865.1761,
    "train_samples_per_second": 23.117,
    "train_steps_per_second": 1.445,
    "total_flos": 1.114222151780352e+16,
    "train_loss": 1.7669204229354858,
    "epoch": 2.0
  }
}


## 7. Adapter 재로드와 동일 prompt 비교

메모리의 학습 객체를 제거하고 base model과 해당 profile의 adapter를 새로 로드한다. 비어 있지 않은 출력은 저장·재로드 성공만 뜻하며 사실 정확도 PASS를 뜻하지 않는다.

In [9]:
del trainer, model
_ = gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

reloaded_base = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=torch.bfloat16,
    local_files_only=True,
).to("cuda")
reloaded_model = PeftModel.from_pretrained(
    reloaded_base,
    ADAPTER_DIR,
    local_files_only=True,
).to("cuda")

adapter_answer = generate_answer(reloaded_model, CHECK_MESSAGES)
reload_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
comparison = {
    "run_profile": RUN_PROFILE,
    "prompt": CHECK_MESSAGES[0]["content"],
    "expected_factual_answer": "4 wooden planks",
    "base_answer": base_answer,
    "adapter_answer": adapter_answer,
    "reload_inference_peak_vram_gib": round(reload_peak_gib, 3),
}
(OUTPUT_DIR / "inference_comparison.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(comparison, ensure_ascii=False, indent=2))
assert adapter_answer, "재로드한 adapter 모델이 빈 응답을 생성했습니다."

NameError: name 'trainer' is not defined

## 8. 결과 해석

- `quick_smoke_128`: 실행과 저장만 확인한다.
- `memorize_128`: train/eval이 같으므로 높은 정확도는 암기 성공일 뿐이다.
- `pilot_1k`, `generalize_10k`: validation loss와 별도의 사실·형식 평가셋을 함께 봐야 한다.
- `mean_token_accuracy`는 completion 토큰의 top-1 일치율이며 문장 또는 task 성공률이 아니다.
- 제작대 질문의 정답은 4 planks다. 이 한 문항의 오답만으로 actor 전체 성능을 판정하지 않되, non-empty 출력만으로도 품질 PASS를 선언하지 않는다.

각 실행 결과는 `research/mineskynet_finetuning/outputs/gemma3_1b_lora_<profile>/`에 분리 저장되고 Git에서 제외된다.